# Retail Sales Analysis and Prediction Using Data Science
**Domain:** Retail | **Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

## Project objective
Analyze retail sales data, find useful business patterns, visualize performance, and train a model to estimate sales.

### Questions this project answers
1. How do sales change over time?
2. Which product categories and sub-categories generate the most sales?
3. Which regions and customer segments perform best?
4. How are sales and profit related?
5. Can we estimate sales from order characteristics?

> **Dataset note:** This notebook uses the public Sample Superstore dataset if you provide its CSV. If the file is not present, it creates a clearly labeled *synthetic practice dataset* so every section can still run. Synthetic results are demonstrations, not real business findings.

## 1. Install and import libraries
Run the install command only if imports fail.

In [ ]:
# If needed, run this in a separate notebook cell:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme()
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
RANDOM_STATE = 42

## 2. Load the dataset
Download a Sample Superstore CSV and put it beside this notebook, named `SampleSuperstore.csv`. Common versions have columns such as `Order Date`, `Sales`, `Profit`, `Category`, `Sub-Category`, `Region`, and `Segment`.

If your file has different column names, review `df.columns` and update the column mapping in the next section.

In [ ]:
DATA_PATH = Path("SampleSuperstore.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE = "Provided SampleSuperstore.csv"
else:
    # Synthetic fallback dataset: used only so the notebook can run without an external file.
    rng = np.random.default_rng(RANDOM_STATE)
    n = 1200
    order_dates = pd.to_datetime("2023-01-01") + pd.to_timedelta(
        rng.integers(0, 730, n), unit="D"
    )
    categories = np.array(["Technology", "Furniture", "Office Supplies"])
    subcats = {
        "Technology": ["Phones", "Accessories", "Machines"],
        "Furniture": ["Chairs", "Tables", "Bookcases"],
        "Office Supplies": ["Paper", "Binders", "Storage", "Art"]
    }
    category = rng.choice(categories, n, p=[0.32, 0.28, 0.40])
    sub_category = [rng.choice(subcats[c]) for c in category]
    region = rng.choice(["West", "East", "Central", "South"], n)
    segment = rng.choice(["Consumer", "Corporate", "Home Office"], n, p=[0.52, 0.30, 0.18])
    quantity = rng.integers(1, 8, n)
    discount = rng.choice([0, 0.1, 0.2, 0.3, 0.5], n, p=[0.42, 0.20, 0.18, 0.15, 0.05])
    base_price = np.select(
        [category == "Technology", category == "Furniture"],
        [rng.uniform(40, 450, n), rng.uniform(25, 350, n)],
        default=rng.uniform(3, 80, n)
    )
    sales = np.maximum(1, base_price * quantity * (1 - discount) + rng.normal(0, 12, n))
    profit = sales * rng.uniform(0.04, 0.24, n) - sales * discount * 0.22 + rng.normal(0, 8, n)
    df = pd.DataFrame({
        "Order Date": order_dates,
        "Category": category,
        "Sub-Category": sub_category,
        "Region": region,
        "Segment": segment,
        "Quantity": quantity,
        "Discount": discount,
        "Sales": sales.round(2),
        "Profit": profit.round(2)
    })
    DATA_SOURCE = "Synthetic practice data (fallback; not real-world evidence)"

print("Dataset source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect and standardize column names

In [ ]:
print("Original columns:")
print(df.columns.tolist())

# Normalize common column names from different Superstore CSV versions.
rename_candidates = {
    "Order Date": ["Order Date", "OrderDate", "order_date"],
    "Sales": ["Sales", "sales", "Revenue"],
    "Profit": ["Profit", "profit"],
    "Category": ["Category", "category"],
    "Sub-Category": ["Sub-Category", "Sub Category", "subcategory"],
    "Region": ["Region", "region"],
    "Segment": ["Segment", "segment"],
    "Quantity": ["Quantity", "quantity", "Qty"],
    "Discount": ["Discount", "discount"]
}
rename_map = {}
for standard, candidates in rename_candidates.items():
    for candidate in candidates:
        if candidate in df.columns:
            rename_map[candidate] = standard
            break
df = df.rename(columns=rename_map)

required = ["Sales"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required column(s): {missing_required}. Please check your CSV.")

# Add optional columns if a source dataset does not contain them.
for col, default in {
    "Order Date": pd.NaT, "Profit": np.nan, "Category": "Unknown",
    "Sub-Category": "Unknown", "Region": "Unknown", "Segment": "Unknown",
    "Quantity": np.nan, "Discount": np.nan
}.items():
    if col not in df.columns:
        df[col] = default

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
for col in ["Sales", "Profit", "Quantity", "Discount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.info())
display(df.describe(include="all").T)

## 4. Data cleaning

In [ ]:
before_rows = len(df)
df = df.drop_duplicates().copy()

# Remove rows where the target sales value is missing or invalid.
df = df.dropna(subset=["Sales"])
df = df[df["Sales"] >= 0].copy()

# Create date features. Invalid/missing dates remain missing and are excluded from time plots.
df["Order_Year"] = df["Order Date"].dt.year
df["Order_Month"] = df["Order Date"].dt.month
df["Order_Month_Name"] = df["Order Date"].dt.month_name()
df["Order_YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)

print(f"Rows before cleaning: {before_rows:,}")
print(f"Rows after cleaning:  {len(df):,}")
print(f"Duplicate rows removed: {before_rows - len(df):,}")
print("\nMissing values after cleaning:")
display(df.isna().sum().to_frame("Missing values"))

## 5. Key performance indicators (KPIs)

In [ ]:
total_sales = df["Sales"].sum()
total_orders = df["Order Date"].nunique() if df["Order Date"].notna().any() else len(df)
average_sale = df["Sales"].mean()
total_profit = df["Profit"].sum(min_count=1)
total_units = df["Quantity"].sum(min_count=1)

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Average Sales per Row", "Total Profit", "Total Quantity", "Rows in Dataset"],
    "Value": [total_sales, average_sale, total_profit, total_units, len(df)]
})
display(kpis)

## 6. Exploratory Data Analysis (EDA)

### 6.1 Sales over time

In [ ]:
monthly_sales = (
    df.dropna(subset=["Order Date"])
      .set_index("Order Date")
      .resample("MS")["Sales"].sum()
)
if not monthly_sales.empty:
    plt.figure(figsize=(12, 5))
    monthly_sales.plot(marker="o")
    plt.title("Monthly Sales Trend")
    plt.xlabel("Order month")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()
else:
    print("No valid dates available for time-series analysis.")

### 6.2 Sales by category

In [ ]:
category_sales = df.groupby("Category", dropna=False)["Sales"].sum().sort_values(ascending=False)
plt.figure(figsize=(9, 5))
sns.barplot(x=category_sales.values, y=category_sales.index)
plt.title("Total Sales by Category")
plt.xlabel("Total sales")
plt.ylabel("Category")
plt.tight_layout()
plt.show()
display(category_sales.to_frame("Total Sales"))

### 6.3 Sales by region

In [ ]:
region_sales = df.groupby("Region", dropna=False)["Sales"].sum().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=region_sales.index, y=region_sales.values)
plt.title("Total Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total sales")
plt.tight_layout()
plt.show()
display(region_sales.to_frame("Total Sales"))

### 6.4 Sales and profit by category

In [ ]:
if df["Profit"].notna().any():
    category_profit = df.groupby("Category")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False)
    display(category_profit)
    category_profit.plot(kind="bar", figsize=(10, 5))
    plt.title("Sales and Profit by Category")
    plt.xlabel("Category")
    plt.ylabel("Amount")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Profit data is unavailable in this dataset.")

### 6.5 Discount versus profit

In [ ]:
discount_profit = df.dropna(subset=["Discount", "Profit"])
if not discount_profit.empty:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=discount_profit, x="Discount", y="Profit", hue="Category", alpha=0.7)
    plt.title("Discount vs Profit")
    plt.tight_layout()
    plt.show()
else:
    print("Discount/profit data is insufficient for this plot.")

### 6.6 Correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(9, 6))
    sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Between Numeric Variables")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for a correlation heatmap.")

## 7. Summary findings from the loaded data

In [ ]:
print("DATA-DRIVEN SUMMARY")
print("-------------------")
print(f"Total sales: {df['Sales'].sum():,.2f}")
print(f"Average sales per row: {df['Sales'].mean():,.2f}")

if df["Category"].notna().any():
    top_category = df.groupby("Category")["Sales"].sum().idxmax()
    print(f"Category with highest sales: {top_category}")

if df["Region"].notna().any():
    top_region = df.groupby("Region")["Sales"].sum().idxmax()
    print(f"Region with highest sales: {top_region}")

if df["Profit"].notna().any():
    print(f"Total profit: {df['Profit'].sum():,.2f}")
    least_profitable_category = df.groupby("Category")["Profit"].sum().idxmin()
    print(f"Lowest total-profit category: {least_profitable_category}")

print("\nUse these computed outputs in your report. Do not copy example values from another dataset.")

## 8. Machine learning: predict sales
We will train a Random Forest regression model to predict a transaction's sales using available order characteristics. This is a learning demonstration; results depend on the dataset and features. If the dataset is small or lacks useful predictors, scores may be weak.

**Important:** The target (`Sales`) is excluded from the input features. Date-derived fields and identifiers are not used as direct target proxies.

In [ ]:
model_df = df.copy()
features = ["Category", "Sub-Category", "Region", "Segment", "Quantity", "Discount"]
target = "Sales"

# Retain rows with a target and at least one usable feature.
model_df = model_df.dropna(subset=[target]).copy()
X = model_df[features]
y = model_df[target]

categorical_features = ["Category", "Sub-Category", "Region", "Segment"]
numeric_features = ["Quantity", "Discount"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

if len(model_df) < 20:
    print("Warning: very few rows; metrics may be unreliable.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R² score"],
    "Value": [mae, rmse, r2],
    "Interpretation": [
        "Average absolute prediction error; lower is better",
        "Penalizes larger errors; lower is better",
        "Explained variance relative to a mean baseline; closer to 1 is generally better"
    ]
})
display(metrics)

## 9. Actual versus predicted sales

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=predictions, alpha=0.7)
lims = [
    min(float(np.min(y_test)), float(np.min(predictions))),
    max(float(np.max(y_test)), float(np.max(predictions)))
]
plt.plot(lims, lims, linestyle="--")
plt.title("Actual vs Predicted Sales")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.tight_layout()
plt.show()

## 10. Residual analysis

In [ ]:
residuals = y_test.to_numpy() - predictions
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title("Distribution of Prediction Errors (Residuals)")
plt.xlabel("Actual sales - predicted sales")
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():,.2f}")

## 11. Predict sales for a new example

In [ ]:
new_order = pd.DataFrame([{
    "Category": "Technology",
    "Sub-Category": "Phones",
    "Region": "West",
    "Segment": "Consumer",
    "Quantity": 2,
    "Discount": 0.10
}])

estimated_sales = model.predict(new_order[features])[0]
print(f"Estimated sales for the example order: {estimated_sales:,.2f}")
print("This is a model estimate, not a guaranteed price or business forecast.")

## 12. Conclusion
Use the figures generated by your own notebook to write the final conclusion. Summarize:
- The category and region with the highest sales.
- Whether sales and profit move together in your dataset.
- What the monthly trend shows.
- The model's MAE, RMSE, and R² score, and what they mean.
- One business action supported by the evidence.

### Limitations
- A sample dataset may not represent every retailer or market.
- Correlation does not prove causation.
- A sales model using only category, region, segment, quantity, and discount has limited information.
- Synthetic fallback data must not be presented as genuine business data.
- Model scores should be checked against a baseline and on newer data before real business use.

### Future improvements
- Use a larger, real sales dataset with product cost, customer history, promotions, and holidays.
- Compare Random Forest with Linear Regression and Gradient Boosting.
- Use time-based validation for genuine future forecasting.
- Build a dashboard in Power BI or Streamlit.

## 13. References
- Pandas documentation: https://pandas.pydata.org/docs/
- Matplotlib documentation: https://matplotlib.org/stable/
- Seaborn documentation: https://seaborn.pydata.org/
- Scikit-learn documentation: https://scikit-learn.org/stable/
- Dataset option: search for the public “Sample Superstore” dataset and follow its license/terms.

**Submission checklist**
- [ ] Run every notebook cell from top to bottom.
- [ ] If available, use a genuine dataset and record its source.
- [ ] Save the notebook with outputs.
- [ ] Add screenshots of at least four charts.
- [ ] Write conclusions using your actual outputs.